In [1]:
#The following file paths are all absolute paths. You can replace them with relative paths at runtime, and the files are located in their respective folders.
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
import matplotlib.pyplot as plt
import random
import argparse
import time
from collections import OrderedDict
from copy import copy
# import Learn_Knonlinear as lka
import scipy
import scipy.linalg
from scipy.integrate import odeint
from tqdm import tqdm, trange
import sys
import os
# os.chdir(r'/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
os.chdir(r'D:/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
sys.path.append("utility_LSPN/")
sys.path.append("LSPN_compare/sizeNN_learnmodel_train")
sys.path.append("utility_LSPN/")
try:
    from LSPN_test import LSPN_Mamba
except:
    pass
from Utility import data_collecter
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"

In [3]:
Methods = ["KNonlinear","KNonlinearRNN","KoopmanU",\
            "KoopmanNonlinearA","KoopmanNonlinear",\
                "KNonlinearmamba"]
Method_names = ["KDNN","KRNN","DKUC(no SOC)",\
            "DKAC(no SOC)","DKN(no SOC)","KNonlinearmamba"]

In [3]:
def eval_err(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "DATA/LSPN_compare_sizeNNdata/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method_index==1:
        import Learn_Knonlinear_RNN as lka
    elif method_index==0:
        import Learn_Knonlinear as lka
    # elif method_index==6:
    #     import learn_DKN_SOC_sizeNN as lka
    # elif method_index==5:
    #     import learn_DKAC_SOC_sizeNN as lka
    # elif method_index==4:
    #     import learn_DKUC_SOC_sizeNN as lka
    elif method_index==2:
        import Learn_DKUC_withoutSOC as lka
    elif method_index==3:
        import Learn_DKAC_withoutSOC as lka
    elif method_index==4:
        import Learn_DKN_withoutSOC as lka
    elif method_index==5:
        import Learn_Knonlinear_mamba as lka
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+env_name+"layer{}".format(layer_i)+"_") and file.endswith(".pth"):
            model_path = file  
    Data_collect = data_collecter(env_name)
    udim = Data_collect.udim
    Nstates = Data_collect.Nstates
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=udim)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=udim+Nstates,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim) 
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda") 
    net.load_state_dict(state_dict)
    total_params = sum(p.numel() for p in net.parameters())
    print(f"{method} Total parameters: {total_params}")
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 50
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 10
    max_loss_all = np.zeros((times,steps))
    mean_loss_all = np.zeros((times,steps))
    min_loss_all = np.zeros((times,steps))
    Test_time = np.zeros((times))
    test_loss = []
    test_loss_log = []
    with torch.no_grad():
        for i in trange(times, desc="predicting", unit="times"):
            start_time = time.time()
            test_data = Data_collect.collect_koopman_data(Samples,steps)
            # np.save("DATA/LSPN_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i),test_data)
            max_loss,mean_loss,min_loss = lka.K_loss(test_data,net,udim,Nstate=Nstates)
            max_loss_all[i] = max_loss.reshape(-1)
            mean_loss_all[i] = mean_loss.reshape(-1)
            test_loss.append(mean_loss.reshape(-1))
            test_loss_log.append(np.log10(mean_loss_all[i]))
            min_loss_all[i] = min_loss.reshape(-1)
            end_time = time.time()
            t_cost = end_time - start_time
            Test_time[i] = t_cost
            print(Test_time[i])
    max_mean = np.mean(max_loss_all,axis=0)
    max_std = np.std(max_loss_all,axis=0)
    mean_mean =  np.mean(mean_loss_all,axis=0)
    mean_std =  np.std(mean_loss_all,axis=0)
    min_mean =  np.mean(min_loss_all,axis=0)
    min_std =  np.std(min_loss_all,axis=0)  
    test_loss = np.array(test_loss)
    test_loss_log = np.array(test_loss_log)
    print("test_loss:{}".format(test_loss_log.shape))
    print("test_time:{}".format(Test_time.shape))
    # np.save("DATA/LSPN_compare_drawdata/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([max_mean,max_std,mean_mean,mean_std,min_mean,min_std]))
    # np.save("DATA/LSPN_compare_drawdata/log_"+env_name+"{}".format(method)+"_{}.npy".format(steps),test_loss_log)
    # np.save("DATA/LSPN_compare_drawdata/"+env_name+"{}".format(method)+"_{}.npy".format(steps),test_loss)
    # np.save("DATA/LSPN_compare_drawdata/time_"+env_name+"{}".format(method)+"_{}.npy".format(steps),Test_time)
    return max_mean,max_std,mean_mean,mean_std,min_mean,min_std

In [4]:
suffix = ["Knolinear_SOC_models","KRNN_SOC_models","DKUC_withoutSOC_sizeNN","DKAC_withoutSOC_sizeNN","DKN_withoutSOC_sizeNN","mamba_test6"]
env_name = "DampingPendulum"
steps = 300
for i in [0,1,2,3,4,5]:#0,1,2,3,4,5
    eval_err(suffix[i],env_name,method_index=i,layer_i=4, steps = steps)

KNonlinear
KNonlinear Total parameters: 50306


predicting:  10%|█         | 1/10 [00:01<00:10,  1.20s/times]

1.197187900543213


predicting:  20%|██        | 2/10 [00:02<00:09,  1.17s/times]

1.1418373584747314


...

1.0962255001068115


predicting:  90%|█████████ | 9/10 [00:09<00:01,  1.10s/times]

1.1063947677612305


predicting: 100%|██████████| 10/10 [00:11<00:00,  1.11s/times]

1.0944743156433105
test_loss:(10, 300)
test_time:(10,)


In [4]:
for k in [0,1,2,3,4,5]:
    env_name = "DampingPendulum"
    method = Methods[k]
    steps = 100
    data_all = np.load("DATA/LSPN_compare_drawdata/"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    time_cost = np.load("DATA/LSPN_compare_drawdata/time_"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    Loss_Data = []
    mean_data = []
    for i in range(steps):#15
        data = data_all[:,i]
        # 计算平均值和标准差
        mean_value = sum(data) / len(data)
        std_deviation = (sum((x - mean_value) ** 2 for x in data) / len(data)) ** 0.5

        # 使用科学计数法格式化字符串
        formatted_string = "{:.3e} ± {:.3e}".format(mean_value, std_deviation)
        Loss_Data.append(formatted_string)
        # print(formatted_string)
        mean_data.append(mean_value)
    Loss_Data = np.array(Loss_Data)
    mean_data = np.array(mean_data)
    print(method)
    print(Loss_Data[14],Loss_Data[29],Loss_Data[49],Loss_Data[99])
    # np.save("DATA/LSPN_compare_drawdata/{}_{}step_loss.npy".format(method, steps),Loss_Data)

KNonlinear
1.289e-01 ± 1.128e-02 2.702e-01 ± 2.837e-02 4.564e-01 ± 6.696e-02 7.350e-01 ± 1.033e-01
KNonlinearRNN
2.461e-02 ± 2.762e-03 3.586e-02 ± 5.049e-03 5.186e-02 ± 9.979e-03 2.673e-02 ± 2.234e-03
KoopmanU
1.729e-01 ± 2.320e-02 7.387e-01 ± 3.365e-02 1.524e+00 ± 5.564e-02 1.771e+00 ± 1.235e-01
KoopmanNonlinearA
5.032e-02 ± 3.954e-03 2.860e-01 ± 3.356e-02 9.083e-01 ± 9.626e-02 1.612e+00 ± 1.420e-01
KoopmanNonlinear
1.031e-01 ± 9.826e-03 1.609e-01 ± 1.271e-02 2.717e-01 ± 2.893e-02 4.694e-01 ± 8.929e-02
KNonlinearmamba
7.738e-04 ± 9.612e-05 9.132e-04 ± 1.472e-04 1.459e-03 ± 2.744e-04 8.070e-04 ± 1.435e-04


In [8]:
time_predict = []
for k in [0,1,2,3,4,5]:
    env_name = "DampingPendulum"
    method = Methods[k]
    steps = 300
    # data_all = np.load("DATA/LSPN_compare_drawdata/"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    data_all = np.load("DATA/LSPN_compare_drawdata/time_"+env_name+"{}".format(method)+"_{}.npy".format(steps))
    time_predict.append(data_all)
    Loss_Data = []
    mean_data = []
    # for i in range(steps):#15
    data = data_all[:]
    # 计算平均值和标准差
    mean_value = sum(data) / len(data)
    std_deviation = (sum((x - mean_value) ** 2 for x in data) / len(data)) ** 0.5

    # 使用科学计数法格式化字符串
    formatted_string = "{:.3e} ± {:.3e}".format(mean_value, std_deviation)
    Loss_Data.append(formatted_string)
    print(formatted_string)
    mean_data.append(mean_value)
    Loss_Data = np.array(Loss_Data)
    mean_data = np.array(mean_data)
    # print(Loss_Data[9],Loss_Data[49],Loss_Data[99],Loss_Data[199])
    np.save("DATA/LSPN_compare_drawdata/time{}_{}step_loss.npy".format(method, steps),Loss_Data)
time_predict = np.array(time_predict)
print(time_predict.shape)
np.save("DATA/LSPN_compare_drawdata/timedraw_{}step_loss.npy".format(steps),time_predict)

1.155e+00 ± 1.488e-02
1.133e+00 ± 1.924e-02
1.105e+00 ± 9.310e-03
1.153e+00 ± 1.826e-02
1.153e+00 ± 6.235e-03
1.105e+00 ± 1.289e-02
(6, 10)
